# Improving the BB84 alarm threshold

*Follows `3a_bb84_grid_link.ipynb`, which found that a small tap (about 5% of rounds) can hide in line noise.*

The error rate of one sifted bit is **e = f/4 + p(1 − f/2)**: tap fraction f, line noise p. The formula matches the 3a circuit runs: at p = 0.025 and f = 0.5 it gives 0.144, and the circuits measured 0.143. Because the formula is exact, this notebook can simulate millions of bits quickly.

Two questions:
1. **How does the smallest catchable tap depend on how many bits the alarm looks at?**
2. **When line noise drifts (temperature, wind on aerial fiber), does a Kalman filter + CUSUM beat a fixed threshold?** Every detector is tuned to the same false-alarm rate, so the comparison is fair.

In [ ]:
import sys, os, json
sys.path.insert(0, "network_design")
import numpy as np, matplotlib.pyplot as plt
from alarm_study import qber, window_sweep, baseline_path, detectors, events, study_calibrated, calibrate
print("check vs 3a circuits: formula at p=0.025, f=0.5 ->", round(qber(0.025, 0.5), 4), "(circuits: 0.143)")

## 1. Bits per decision vs. smallest tap caught

Each alarm level is set for a 0.2% false-alarm rate (3σ above a quiet link). The smallest catchable tap is the one caught 95% of the time. Prediction: f_min ≈ 4(3 + 1.645)·σ / (1 − 2p), with σ = √(p(1 − p)/N).

In [ ]:
ws = window_sweep()
N = [r["N"] for r in ws]
plt.figure(figsize=(6.5, 4))
plt.loglog(N, [100 * r["f_min"] for r in ws], "o-", label="simulated")
plt.loglog(N, [100 * r["f_min_formula"] for r in ws], "--", label="formula")
plt.xlabel("sifted bits per alarm decision (N)"); plt.ylabel("smallest tap caught 95% of the time (% of rounds)")
plt.title("Smallest catchable tap halves each time N grows 4x (line noise 2.5%)", fontsize=9.5); plt.legend()
plt.grid(True, which="both", alpha=.3); plt.tight_layout(); plt.savefig("3d_window.png", dpi=150); plt.show()
for r in ws: print(f"N = {r['N']:>7,}: smallest tap {100*r['f_min']:.2f}%  (formula {100*r['f_min_formula']:.2f}%)")

## 2. Drifting line noise: fixed threshold vs. Kalman vs. Kalman + CUSUM

Line noise drifts by ±0.8 percentage points (a slow swing plus a random walk). Each batch holds 4,000 sifted bits. A tap starts at batch 900 and stays on.

- **Fixed threshold:** baseline learned once, alarm at baseline + zσ.
- **Kalman:** follows the drifting baseline and alarms when one batch jumps above it.
- **Kalman + CUSUM:** same baseline, but adds up small excess errors over many batches. The baseline is held still while evidence builds, so a slow tap isn't absorbed into it.

Each detector's setting below was chosen so a quiet drifting link raises **about 1 false alarm per 1,000 batches**. `calibrate()` reproduces these values in a few minutes.

In [ ]:
cfg = {"fixed": {"z_fixed": 13.5}, "kalman": {"z_fixed": 4.0}, "kalman_cusum": {"h_sigma": 15.0}}
taps = (0.005, 0.01, 0.02, 0.05)
res = study_calibrated(cfg, f_taps=taps, runs=150)
lab = {"fixed": "Fixed threshold", "kalman": "Kalman", "kalman_cusum": "Kalman + CUSUM"}
plt.figure(figsize=(6.5, 4))
for k in cfg: plt.plot([100 * t for t in taps], [100 * res[t][k]["caught"] for t in taps], "o-", label=lab[k])
plt.xlabel("tap size (% of rounds)"); plt.ylabel("% of runs caught within 300 batches"); plt.ylim(-3, 103)
plt.title("Drifting line noise, every detector at ~1 false alarm per 1,000 batches", fontsize=9.5); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig("3d_detectors.png", dpi=150); plt.show()
for t in taps: print(f"tap {100*t:.1f}%: " + " | ".join(f"{lab[k]}: caught {100*v['caught']:.0f}%, false alarms {v['false_alarms_per_1000']}/1000, median delay {v['median_delay']}" for k, v in res[t].items()))

In [ ]:
# One example run: drifting baseline, a 5% tap from batch 900, and where each detector alarms.
# Seed 1 was picked as a typical run: no false alarms before the tap, and Kalman + CUSUM catching it within a few
# batches, which matches the 100% catch rate and ~3-batch median delay above. The fixed threshold misses it.
rng = np.random.default_rng(1); T, Nb, t_tap, f = 1500, 4000, 900, 0.05
p = baseline_path(T, rng, gusts=False); fv = np.where(np.arange(T) >= t_tap, f, 0.0)
z = rng.binomial(Nb, qber(p, fv)) / Nb
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(100 * z, lw=.6, color="#9aa5b1", label="measured QBER per batch")
ax.plot(100 * qber(p, fv), lw=1.5, color="#2b2d42", label="true QBER")
ax.axvline(t_tap, color="tab:red", ls="--", label="tap starts (5%)")
for i, (k, c) in enumerate([("fixed", "#4c6ef5"), ("kalman", "#4a7c59"), ("kalman_cusum", "#e07a1f")]):
    ev = events(detectors(z, Nb, **cfg[k])[k])
    ax.plot(ev, np.full(len(ev), 100 * z.max() * 1.03 + 0.15 * i), "v", color=c, ms=7, label=f"{lab[k]} alarm")
ax.set_xlabel("batch"); ax.set_ylabel("QBER (%)"); ax.legend(fontsize=7.5, ncol=3, loc="lower left")
ax.set_title("Example run: drifting noise with a 5% tap (typical run, see code comment)", fontsize=9.5); plt.tight_layout(); plt.savefig("3d_example.png", dpi=150); plt.show()

## 3. Wind gusts: the limit of QBER alone

Short gusts on aerial fiber raise QBER by 1–2 points for about 15 batches. EPB saw exactly this in its field test.

In [ ]:
resg = study_calibrated(cfg, f_taps=(0.0, 0.02), runs=150, gusts=True)
for t, d in resg.items():
    print(f"tap {100*t:.0f}% with gusts: " + " | ".join(f"{lab[k]}: false alarms {v['false_alarms_per_1000']}/1000, alarms in tap window {100*v['caught']:.0f}%" for k, v in d.items()))

## 4. Adding a classical fiber monitor as a second signal

A classical monitor (vibration/polarization sensing plus OTDR) gives two extra signals:
- **Disturbance flag:** sees about 90% of wind gusts, with rare false flags.
- **Loss step:** sees about 50% of taps being installed. A careful low-loss tap can avoid it.

The fused detector uses them two ways:
- **Gust hold:** while a disturbance is flagged, CUSUM adds no evidence and the baseline holds still. Both resume afterward, so a tap that outlasts the gust is still caught.
- **OTDR boost:** after a loss step, the CUSUM alarm level drops to 40% for 200 batches.

The QBER-only detector is run at its quietest setting (h = 35). Raising h further barely lowers its false alarms, because the gusts are large enough to trip it at almost any setting.

In [ ]:
from alarm_study import gusty_path, monitor, fused_cusum
def compare(f, runs=200, seed=21, p_see_tap=0.5, h_q=35):
    rng = np.random.default_rng(seed); T, N, t_tap = 1500, 4000, 900
    res = {"QBER only": [0, 0, []], "Fused with fiber monitor": [0, 0, []]}
    for _ in range(runs):
        p, g = gusty_path(T, rng); fv = np.where(np.arange(T) >= t_tap, f, 0.0); z = rng.binomial(N, qber(p, fv)) / N
        env, otdr = monitor(g, t_tap if f > 0 else None, T, rng, p_see_tap=p_see_tap)
        for k, a in zip(res, (detectors(z, N, h_sigma=h_q)["kalman_cusum"], fused_cusum(z, N, env, otdr))):
            ev = events(a); res[k][0] += int(((ev >= 100) & (ev < t_tap)).sum())
            af = ev[(ev >= t_tap) & (ev < t_tap + 300)] - t_tap
            if f > 0 and af.size: res[k][1] += 1; res[k][2].append(int(af[0]))
    return {k: dict(fa=1000 * v[0] / (runs * (t_tap - 100)), caught=v[1] / runs, delay=float(np.median(v[2])) if v[2] else None) for k, v in res.items()}

taps4 = (0.0, 0.01, 0.02, 0.05)
R4 = {f: compare(f) for f in taps4}
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for k, c in (("QBER only", "#4c6ef5"), ("Fused with fiber monitor", "#e07a1f")):
    ax[0].plot([100 * f for f in taps4[1:]], [100 * R4[f][k]["caught"] for f in taps4[1:]], "o-", color=c, label=f"{k} ({R4[0.0][k]['fa']:.1f} false alarms/1000)")
    ax[1].plot([100 * f for f in taps4[1:]], [R4[f][k]["delay"] for f in taps4[1:]], "o-", color=c, label=k)
ax[0].set_xlabel("tap size (% of rounds)"); ax[0].set_ylabel("% of runs caught within 300 batches"); ax[0].set_ylim(0, 103); ax[0].legend(fontsize=8)
ax[1].set_xlabel("tap size (% of rounds)"); ax[1].set_ylabel("median batches to alarm"); ax[1].legend(fontsize=8)
fig.suptitle("Drifting noise with wind gusts: QBER alone vs. fused with a classical fiber monitor", fontsize=10)
plt.tight_layout(); plt.savefig("3d_fusion.png", dpi=150); plt.show()
for f in taps4: print(f"tap {100*f:.0f}%: " + " | ".join(f"{k}: false alarms {v['fa']:.2f}/1000, caught {100*v['caught']:.0f}%, delay {v['delay']}" for k, v in R4[f].items()))
c = compare(0.02, p_see_tap=0.0)
print("careful attacker (OTDR never sees the tap), 2% tap: " + " | ".join(f"{k}: caught {100*v['caught']:.0f}%, delay {v['delay']}" for k, v in c.items()))

### Baseline: the classical monitor alone (no QKD)

What does QKD's error-rate check add over a cheap classical monitor? The monitor alone only alarms on a loss step, so it catches a tap only if it sees the installation, whatever the tap's size. A careful low-loss tap evades it completely. Its own false alarms are taken as 1 spurious loss step per 2,000 batches (an assumption).

In [ ]:
def monitor_only(f, runs=200, seed=21, p_see_tap=0.5, p_false_step=0.0005):
    rng = np.random.default_rng(seed); T, t_tap = 1500, 900; fa = det = 0
    for _ in range(runs):
        p, g = gusty_path(T, rng)
        _, otdr = monitor(g, t_tap if f > 0 else None, T, rng, p_see_tap=p_see_tap)
        otdr |= rng.random(T) < p_false_step
        ev = np.flatnonzero(otdr); fa += int(((ev >= 100) & (ev < t_tap)).sum())
        if f > 0 and ((ev >= t_tap) & (ev < t_tap + 300)).any(): det += 1
    return dict(fa=1000 * fa / (runs * (t_tap - 100)), caught=det / runs)
for label, kw in [("typical tap (monitor sees 50%)", {}), ("careful tap (monitor sees 0%)", {"p_see_tap": 0.0})]:
    m = monitor_only(0.02, **kw); q = compare(0.02, **kw)
    print(f"2% {label}: monitor only {100*m['caught']:.0f}% caught ({m['fa']:.2f} false alarms/1000) | "
          f"QBER only {100*q['QBER only']['caught']:.0f}% | fused {100*q['Fused with fiber monitor']['caught']:.0f}%")

## What this shows

- **More bits per decision is the biggest lever.** The smallest catchable tap scales as 1/√N. At the kbps key rates EPB measured, a window of a few minutes holds hundreds of thousands of sifted bits, enough to catch taps under 1% of rounds.
- **On drifting noise, Kalman + CUSUM catches far more at the same false-alarm rate.** At a 2% tap: fixed threshold catches about 10%, Kalman about 30%, Kalman + CUSUM about 65%. At 5%: about 20%, 95% and 100%. A fixed threshold must sit high above the drift, so it misses small taps.
- **QBER alone can't tell a wind gust from a tap.** Every detector raises more false alarms when gusts are present, and tightening the QBER alarm barely helps.
- **A classical fiber monitor fixes most of this.** Fused with it, the detector raises about 40% fewer false alarms *and* catches more taps, much faster (2% tap: about 83% vs 57%, median 7 vs 59 batches). The gain holds even when the monitor misses the tap itself (about 67% vs 57%).
- **QKD's error-rate check adds what the monitor can't: detection that doesn't depend on the attacker being careless.** The monitor alone catches only taps it sees being installed (about 50%, or 0% for a careful low-loss tap). QKD's check catches taps by what they do to the photons, so it still works against the careful attacker, and fusing the two is best.
- **Remaining weakness: an attacker who taps only during gusts** is held along with the gust. The fix is a policy, not better detection: **discard key material generated during flagged disturbance windows**, so a tap timed to gusts captures nothing usable.
- **Assumptions:** the drift and gust model is illustrative, shaped like EPB's reported behavior but not fitted to it. Batches of 4,000 sifted bits; Eve uses intercept-resend.